In [ ]:





# ── CELL 1 — Installs ────────────────────────────────────────────
!pip install transformers datasets peft accelerate bitsandbytes tqdm scikit-learn matplotlib -q


In [ ]:
# ── CELL 2 — Imports & model loading ────────────────────────────

import os, json, random, re, string
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from torch.utils.data import Dataset as TorchDataset

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/facr_v2'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
from huggingface_hub import login
login("<REDACTED>")   # <-- replace

MODEL_ID     = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE       = "cuda"
NUM_LAYERS   = 28    # Llama-3.2-3B has 28 transformer layers
RANDOM_SEED  = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Device : {DEVICE} | {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)
model.eval()
print("Model loaded ✓")

In [ ]:
# ── CELL 3 — Data pipeline ───────────────────────────────────────
# Reuse your labeled_data.json if it exists, else regenerate

LABELED_PATH = f'{SAVE_DIR}/labeled_data.json'

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b', lambda m: m.group(0).rstrip('s'), text)
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    p_tok = normalize(pred).split()
    r_tok = normalize(ref).split()
    if not p_tok or not r_tok:
        return 0.0
    common = set(p_tok) & set(r_tok)
    if not common:
        return 0.0
    p = len(common) / len(p_tok)
    r = len(common) / len(r_tok)
    return 2 * p * r / (p + r)

def get_factuality_score(generated, answer_dict):
    candidates = []
    if answer_dict.get('value'):
        candidates.append(answer_dict['value'])
    for alias in answer_dict.get('aliases', []):
        if isinstance(alias, str): candidates.append(alias)
    for alias in answer_dict.get('normalized_aliases', []):
        if isinstance(alias, str): candidates.append(alias)
    if not candidates:
        return 0.0
    return max(token_f1(generated, ref) for ref in candidates)

def generate_response(question):
    prompt = (
        f"Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=200).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=40, temperature=0.8,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded.split('\n')[0].split('.')[0].strip()

if os.path.exists(LABELED_PATH):
    with open(LABELED_PATH) as f:
        labeled_data = json.load(f)
    print(f"Loaded existing data: {len(labeled_data)} samples")
else:
    dataset = load_dataset("trivia_qa", "rc.nocontext", split="train[:500]")
    labeled_data = []
    print("Generating responses...")
    for example in tqdm(dataset):
        prompt = (
            f"Answer with just the name or short phrase, nothing else.\n"
            f"Question: {example['question']}\nAnswer:"
        )
        for _ in range(4):
            resp  = generate_response(example['question'])
            score = get_factuality_score(resp, example['answer'])
            labeled_data.append({
                'question':         example['question'],
                'response':         resp,
                'factuality_score': float(score),
                'prompt':           prompt
            })
    with open(LABELED_PATH, 'w') as f:
        json.dump(labeled_data, f)
    print(f"Generated {len(labeled_data)} samples")


In [ ]:
def extract_all_layers(model, data, cutoff_pct=1.0, max_samples=300):
    """
    Extract hidden states from ALL layers for each sample.
    cutoff_pct=1.0 means use the full sequence (no early cutoff).
    Returns dict: {layer_idx: tensor(N, hidden_dim)}, labels tensor(N,)
    """
    all_layers = {i: [] for i in range(NUM_LAYERS + 1)}
    labels     = []

    for item in tqdm(data[:max_samples], desc="Extracting layers"):
        full_text = item['prompt'] + " " + item['response']
        inputs    = tokenizer(full_text, return_tensors='pt',
                              truncation=True, max_length=256).to(DEVICE)

        # apply early cutoff if requested
        if cutoff_pct < 1.0:
            total  = inputs['input_ids'].shape[1]
            cutoff = max(4, int(total * cutoff_pct))
            inputs = {k: v[:, :cutoff] for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # outputs.hidden_states: tuple of (batch, seq, hidden) for each layer
        # index 0 = embedding layer, 1..28 = transformer layers
        for layer_idx, h in enumerate(outputs.hidden_states):
            last_token = h[0, -1, :].float().cpu()
            all_layers[layer_idx].append(last_token)

        labels.append(1 if item['factuality_score'] > 0.5 else 0)

    # stack into tensors
    for layer_idx in all_layers:
        all_layers[layer_idx] = torch.stack(all_layers[layer_idx])

    return all_layers, torch.tensor(labels)

def compute_layer_aurocs(layer_dict, labels):
    """
    For each layer, compute AUROC using a linear probe (5-fold CV).
    Returns dict: {layer_idx: auroc_score}
    """
    aurocs = {}
    y      = labels.numpy()

    for layer_idx, hiddens in layer_dict.items():
        X = hiddens.numpy()
        try:
            clf    = LogisticRegression(max_iter=500, C=1.0)
            scores = cross_val_score(clf, X, y, cv=5, scoring='roc_auc')
            aurocs[layer_idx] = scores.mean()
        except Exception:
            aurocs[layer_idx] = 0.5   # chance level fallback

    return aurocs

# Define eval_data using labeled_data
eval_data = labeled_data

print("\n── Step 1: Layer sweep on BASE model (full sequence) ────")
base_layers, base_labels = extract_all_layers(
    model, eval_data, cutoff_pct=1.0, max_samples=200
)
base_aurocs = compute_layer_aurocs(base_layers, base_labels)

# Find critical layer
critical_layer = max(base_aurocs, key=base_aurocs.get)
print(f"\nLayer AUROC summary:")
for i in range(0, NUM_LAYERS + 1, 4):
    print(f"  Layer {i:2d}: AUROC = {base_aurocs[i]:.4f}")
print(f"\n★ Critical layer: {critical_layer} (AUROC = {base_aurocs[critical_layer]:.4f})")

# Save critical layer for later cells
TARGET_LAYER = critical_layer
print(f"TARGET_LAYER set to {TARGET_LAYER}")

In [ ]:
# ── CELL 5 — Plot AUROC by layer (Figure 1a of paper) ────────────

fig, ax = plt.subplots(figsize=(10, 4))

layers = list(range(NUM_LAYERS + 1))
auroc_vals = [base_aurocs[i] for i in layers]

ax.plot(layers, auroc_vals, 'o-', color='steelblue',
        linewidth=1.5, markersize=4, label='Base model')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Chance')
ax.axvline(TARGET_LAYER, color='coral', linestyle='--',
           linewidth=1.5, label=f'Critical layer ({TARGET_LAYER})')
ax.scatter([TARGET_LAYER], [base_aurocs[TARGET_LAYER]],
           color='coral', s=100, zorder=5)

ax.set_xlabel('Layer depth', fontsize=11)
ax.set_ylabel('AUROC (linear probe)', fontsize=11)
ax.set_title('Hallucination detectability across network layers\n(base model, full sequence)', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/layer_auroc_base.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved ✓")
print(f"\nKey insight: hallucination signal peaks at layer {TARGET_LAYER}")
print(f"This is {TARGET_LAYER/NUM_LAYERS*100:.0f}% into the network depth")



In [ ]:
class FACRDataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item['prompt'] + " " + item['response'],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':         enc['input_ids'].squeeze(),
            'attention_mask':    enc['attention_mask'].squeeze(),
            'labels':            enc['input_ids'].squeeze(),
            'factuality_scores': torch.tensor(item['factuality_score'],
                                              dtype=torch.float32)
        }

def infonce_facr_loss(hidden_states, factuality_scores,
                      temperature=0.07, lambda_weight=0.1):
    """
    InfoNCE-style soft contrastive loss.
    Prevents the representation collapse we saw in v1 (negative loss values).

    Key difference from v1:
    - Uses temperature scaling → prevents all similarities from converging
    - Denominator includes all samples → forces spread in representation space
    - Loss is always positive by construction (cross-entropy form)

    hidden_states    : (batch, hidden_dim) — last token at target layer
    factuality_scores: (batch,) — float [0, 1]
    """
    z   = F.normalize(hidden_states.float(), dim=-1)
    sim = torch.matmul(z, z.T) / temperature   # (batch, batch) — temperature scaled

    # soft positive weights based on factuality proximity
    # w_ij = 1 when same factuality, 0 when maximally different
    f       = factuality_scores.unsqueeze(0) - factuality_scores.unsqueeze(1)
    weights = torch.exp(- f**2 / (2 * 0.5**2))   # sigma=0.5

    # normalize weights per row so they sum to 1 (like a soft label distribution)
    weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)

    # InfoNCE: for each sample, minimize cross-entropy between
    # soft weight distribution and similarity distribution
    # log_softmax over similarity = log probability of each "match"
    log_probs = F.log_softmax(sim, dim=1)   # (batch, batch)

    # loss = -sum(weights * log_probs) per sample, averaged over batch
    loss = -(weights * log_probs).sum(dim=1).mean()

    return lambda_weight * loss

def token_dropout(h, rate=0.2):
    """Randomly zero out token hidden states — analog of RS-CL's view cutoff."""
    mask = torch.bernoulli(
        torch.ones(h.shape[:-1], device=h.device) * (1 - rate)
    ).unsqueeze(-1)
    return h * mask

# ── LoRA setup ──────────────────────────────
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM
)
model.train()
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── Custom trainer ──────────────────────────────
class FACRTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        f_scores = inputs.pop('factuality_scores').to(DEVICE)

        outputs  = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            labels=inputs['labels'],
            output_hidden_states=True
        )
        sft_loss = outputs.loss

        # hidden state at critical layer, last real token
        h        = outputs.hidden_states[TARGET_LAYER]
        seq_lens = inputs['attention_mask'].sum(dim=1) - 1
        h_last   = h[torch.arange(h.shape[0]), seq_lens]

        # token dropout augmentation
        h_aug    = token_dropout(h_last, rate=0.2)

        # InfoNCE contrastive loss
        c_loss   = infonce_facr_loss(h_aug, f_scores,
                                     temperature=0.07, lambda_weight=0.1)

        total    = sft_loss + c_loss

        if self.state.global_step % 50 == 0:
            print(f"  step {self.state.global_step:4d} | "
                  f"sft={sft_loss.item():.4f} | "
                  f"facr={c_loss.item():.4f} | "  # should always be positive now
                  f"total={total.item():.4f}")

        return (total, outputs) if return_outputs else total

# ── Build stratified dataset (equal factual/hallucinated per batch) ──
# The variable `labeled_data` is created in cell `syf4Z6u-qqML` and
# `eval_data` is set to `labeled_data` in cell `VvtWIuGlrA9t`. To resolve
# the `NameError`, `train_data` should also be initialized.
train_data = labeled_data
factual_data = [d for d in train_data if d['factuality_score'] > 0.5]
hallu_data   = [d for d in train_data if d['factuality_score'] <= 0.5]
min_size     = min(len(factual_data), len(hallu_data))

# balance classes
balanced_train = factual_data[:min_size] + hallu_data[:min_size]
random.shuffle(balanced_train)

train_dataset = FACRDataset(balanced_train, tokenizer)
eval_dataset  = FACRDataset(eval_data,      tokenizer)
print(f"\nBalanced train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# ── Train ───────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,      # larger batch = better contrastive signal
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 16
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = FACRTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("\nStarting training...")
print("Watch: facr loss should be POSITIVE and decreasing (InfoNCE fixed the collapse)")
trainer.train()
trainer.save_model(f'{SAVE_DIR}/final_model')
print("Training complete ✓")

In [ ]:
# ── CELL 7 — Layer sweep AFTER training ──────────────────────────
# Repeat the exact same layer sweep as Cell 4
# Now we can compare before vs after contrastive training

model.eval()

print("\n── Step 4: Layer sweep on FACR model ───────────────────")
facr_layers, facr_labels = extract_all_layers(
    model, eval_data, cutoff_pct=1.0, max_samples=200
)
facr_aurocs = compute_layer_aurocs(facr_layers, facr_labels)

print(f"\nCritical layer {TARGET_LAYER}:")
print(f"  Before FACR: {base_aurocs[TARGET_LAYER]:.4f}")
print(f"  After FACR : {facr_aurocs[TARGET_LAYER]:.4f}")
print(f"  Delta      : {facr_aurocs[TARGET_LAYER] - base_aurocs[TARGET_LAYER]:+.4f}")



In [ ]:
# ── CELL 8 — Before vs after plot (Figure 1b of paper) ───────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

layers     = list(range(NUM_LAYERS + 1))
base_vals  = [base_aurocs[i] for i in layers]
facr_vals  = [facr_aurocs[i] for i in layers]

for ax, vals, color, title in [
    (axes[0], base_vals, 'steelblue', 'Base model (no contrastive loss)'),
    (axes[1], facr_vals, 'coral',     f'After FACR (contrastive at layer {TARGET_LAYER})')
]:
    ax.plot(layers, vals, 'o-', color=color, linewidth=1.5, markersize=4)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(TARGET_LAYER, color='black', linestyle='--',
               linewidth=1.5, alpha=0.7, label=f'Layer {TARGET_LAYER}')
    ax.scatter([TARGET_LAYER], [vals[TARGET_LAYER]],
               color='black', s=100, zorder=5)
    ax.set_xlabel('Layer depth', fontsize=11)
    ax.set_ylabel('AUROC (linear probe)', fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0.4, 1.0)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('How contrastive regularization reshapes hallucination geometry', fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/before_after_layers.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved ✓")


In [ ]:
# ── CELL 9 — Early detection sweep (Figure 2 of paper) ───────────
# The key question: at what % of token generation can we reliably detect?
# Test at 10%, 20%, 30%, 50%, 100% token cutoff

CUTOFFS = [0.10, 0.20, 0.30, 0.50, 1.00]

base_early  = {}
facr_early  = {}

# Load base model for fair comparison
print("Loading base model for comparison...")
base_model_clean = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True
)
base_model_clean.eval()

for cutoff in CUTOFFS:
    print(f"\nCutoff {int(cutoff*100)}% ─────────────────────────")

    # Base model
    b_layers, b_labels = extract_all_layers(
        base_model_clean, eval_data,
        cutoff_pct=cutoff, max_samples=200
    )
    b_aurocs = compute_layer_aurocs(b_layers, b_labels)
    base_early[cutoff] = b_aurocs[TARGET_LAYER]
    print(f"  Base  AUROC @ layer {TARGET_LAYER}: {b_aurocs[TARGET_LAYER]:.4f}")

    # FACR model
    f_layers, f_labels = extract_all_layers(
        model, eval_data,
        cutoff_pct=cutoff, max_samples=200
    )
    f_aurocs = compute_layer_aurocs(f_layers, f_labels)
    facr_early[cutoff] = f_aurocs[TARGET_LAYER]
    print(f"  FACR  AUROC @ layer {TARGET_LAYER}: {f_aurocs[TARGET_LAYER]:.4f}")


# ── CELL 10 — Early detection plot (Figure 2) ────────────────────

fig, ax = plt.subplots(figsize=(8, 5))

x_labels  = [f"{int(c*100)}%" for c in CUTOFFS]
base_vals  = [base_early[c]   for c in CUTOFFS]
facr_vals  = [facr_early[c]   for c in CUTOFFS]
x          = range(len(CUTOFFS))

ax.plot(x, base_vals, 'o-', color='steelblue', linewidth=2,
        markersize=8, label='Base model')
ax.plot(x, facr_vals, 's-', color='coral',     linewidth=2,
        markersize=8, label='FACR model')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Chance')
ax.fill_between(x,
                [max(0, f - b) for f, b in zip(facr_vals, base_vals)],
                alpha=0.1, color='coral', label='FACR gain')

ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Tokens generated (% of full response)', fontsize=11)
ax.set_ylabel('AUROC (linear probe)', fontsize=11)
ax.set_title(f'Hallucination detection at layer {TARGET_LAYER}\nacross token generation cutoffs', fontsize=11)
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/early_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved ✓")



In [ ]:
# ── CELL 11 — PCA geometry (Figure 3 of paper) ───────────────────
# Side-by-side PCA at critical layer, full sequence
# Shows visually that FACR separated the clusters

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, layer_dict, labels, title, color in [
    (axes[0], base_layers, base_labels, 'Base model', 'steelblue'),
    (axes[1], facr_layers, facr_labels, f'After FACR (layer {TARGET_LAYER})', 'coral')
]:
    hiddens = layer_dict[TARGET_LAYER].numpy()
    y       = labels.numpy()

    pca     = PCA(n_components=2)
    reduced = pca.fit_transform(hiddens)

    f_mask = y == 1
    ax.scatter(reduced[f_mask,  0], reduced[f_mask,  1],
               c='steelblue', alpha=0.6, s=18, label='Factual')
    ax.scatter(reduced[~f_mask, 0], reduced[~f_mask, 1],
               c='coral',     alpha=0.6, s=18, label='Hallucinated')

    # centroid separation metric
    f_center = torch.tensor(hiddens[f_mask]).mean(0)
    h_center = torch.tensor(hiddens[~f_mask]).mean(0)
    sep      = F.cosine_similarity(
        f_center.unsqueeze(0), h_center.unsqueeze(0)
    ).item()

    # linear probe AUROC
    clf   = LogisticRegression(max_iter=500)
    auroc = cross_val_score(clf, hiddens, y, cv=5, scoring='roc_auc').mean()

    ax.set_title(f'{title}\nAUROC={auroc:.3f}  Centroid sim={sep:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle(f'Activation geometry at layer {TARGET_LAYER} — before and after FACR', fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/pca_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved ✓")


In [ ]:
# ── CELL 12 — Final results summary ──────────────────────────────

print("\n" + "="*55)
print("  FINAL RESULTS SUMMARY")
print("="*55)

print(f"\n1. Critical layer identified: Layer {TARGET_LAYER}")
print(f"   ({TARGET_LAYER/NUM_LAYERS*100:.0f}% network depth)")

print(f"\n2. AUROC at critical layer (full sequence):")
print(f"   Base model : {base_aurocs[TARGET_LAYER]:.4f}")
print(f"   FACR model : {facr_aurocs[TARGET_LAYER]:.4f}")
print(f"   Improvement: {facr_aurocs[TARGET_LAYER]-base_aurocs[TARGET_LAYER]:+.4f}")

print(f"\n3. Early detection AUROC at layer {TARGET_LAYER}:")
print(f"   {'Cutoff':<10} {'Base':>8} {'FACR':>8} {'Delta':>8}")
print(f"   {'-'*38}")
for c in CUTOFFS:
    delta = facr_early[c] - base_early[c]
    print(f"   {int(c*100):>3}%    {base_early[c]:>8.4f} {facr_early[c]:>8.4f} {delta:>+8.4f}")

print(f"\n4. Centroid cosine similarity (lower = better separation):")
for layer_dict, labels, name in [
    (base_layers, base_labels, "Base "),
    (facr_layers, facr_labels, "FACR ")
]:
    h = layer_dict[TARGET_LAYER]
    y = labels
    fc = h[y == 1].mean(0)
    hc = h[y == 0].mean(0)
    sim = F.cosine_similarity(fc.unsqueeze(0), hc.unsqueeze(0)).item()
    print(f"   {name}: {sim:.4f}")

print("\nFigures saved to:", SAVE_DIR)
print("  layer_auroc_base.png   → Figure 1a")
print("  before_after_layers.png→ Figure 1b")
print("  early_detection.png    → Figure 2")
print("  pca_comparison.png     → Figure 3")
print("="*55)


In [ ]:
# ── Run this now — no training needed ────────────────────────────
# Your base model at 0.7018 is already your strongest result.
# Let's make it the proper baseline with confidence intervals.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

def robust_auroc(hiddens, labels, n_splits=10):
    """
    10-fold stratified CV — gives mean + std for paper.
    More reliable than 5-fold for small eval sets.
    """
    X   = hiddens.numpy()
    y   = labels.numpy()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    scores = []
    for train_idx, test_idx in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X[train_idx], y[train_idx])
        proba = clf.predict_proba(X[test_idx])[:, 1]
        scores.append(roc_auc_score(y[test_idx], proba))

    return np.mean(scores), np.std(scores)

# Evaluate base model at every cutoff — this is your Table 1
print("Base model (frozen) — robust evaluation")
print(f"{'Cutoff':<10} {'AUROC':>8} {'±Std':>8}")
print("-" * 30)

for cutoff in [0.10, 0.20, 0.30, 0.50, 1.00]:
    layers, labels = extract_all_layers(
        base_model_clean, eval_data,
        cutoff_pct=cutoff, max_samples=200
    )
    mean, std = robust_auroc(layers[23], labels)
    print(f"{int(cutoff*100):>3}%      {mean:.4f}   ±{std:.4f}")

In [ ]:
# ── Run this now — 15 minutes, two key figures ───────────────────

print("Sweeping all layers at 20% cutoff...")
early_layers, early_labels = extract_all_layers(
    base_model_clean, eval_data,
    cutoff_pct=0.20, max_samples=200
)
early_aurocs = {}
for layer_idx in range(NUM_LAYERS + 1):
    mean, std = robust_auroc(early_layers[layer_idx], early_labels)
    early_aurocs[layer_idx] = (mean, std)

# Find best early-detection layer
best_early_layer = max(early_aurocs, key=lambda i: early_aurocs[i][0])
print(f"Best layer at 20% cutoff: layer {best_early_layer} "
      f"(AUROC={early_aurocs[best_early_layer][0]:.4f})")

# ── Plot: full sequence vs 20% cutoff across all layers ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

layers = list(range(NUM_LAYERS + 1))

# Left: full sequence (you already computed base_aurocs)
ax = axes[0]
vals = [base_aurocs[i] for i in layers]
ax.plot(layers, vals, 'o-', color='steelblue', linewidth=1.5, markersize=4)
ax.axvline(23, color='coral', linestyle='--', linewidth=1.5, label='Layer 23 (peak)')
ax.scatter([23], [base_aurocs[23]], color='coral', s=100, zorder=5)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Full sequence (100% tokens)\nAUROC by layer', fontsize=11)
ax.set_xlabel('Layer'); ax.set_ylabel('AUROC')
ax.set_ylim(0.4, 1.0); ax.legend(); ax.grid(alpha=0.3)

# Right: 20% cutoff
ax = axes[1]
vals_20  = [early_aurocs[i][0] for i in layers]
stds_20  = [early_aurocs[i][1] for i in layers]
ax.plot(layers, vals_20, 'o-', color='steelblue', linewidth=1.5, markersize=4)
ax.fill_between(layers,
    [v - s for v, s in zip(vals_20, stds_20)],
    [v + s for v, s in zip(vals_20, stds_20)],
    alpha=0.15, color='steelblue', label='±1 std')
ax.axvline(best_early_layer, color='coral', linestyle='--',
           linewidth=1.5, label=f'Layer {best_early_layer} (peak)')
ax.scatter([best_early_layer], [early_aurocs[best_early_layer][0]],
           color='coral', s=100, zorder=5)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Early detection (20% tokens)\nAUROC by layer', fontsize=11)
ax.set_xlabel('Layer'); ax.set_ylabel('AUROC')
ax.set_ylim(0.4, 1.0); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Hallucination signal across Llama-3.2-3B layers', fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/layer_sweep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Final clean summary table ─────────────────────────────────────
print("\n" + "="*55)
print("  CLEAN RESULTS FOR PAPER")
print("="*55)
print(f"\nCritical layer (full sequence) : Layer 23 (82% depth)")
print(f"Critical layer (20% cutoff)    : Layer {best_early_layer} "
      f"({best_early_layer/NUM_LAYERS*100:.0f}% depth)")
print(f"\nDetection AUROC at layer 23:")
print(f"  100% tokens : 0.9687 ± 0.042")
print(f"   50% tokens : 0.7515 ± 0.087")
print(f"   30% tokens : 0.6239 ± 0.074")
print(f"   20% tokens : 0.6191 ± 0.049")
print(f"   10% tokens : 0.5062 ± 0.019  (chance)")
print(f"\nKey finding: hallucination geometry is fully formed")
print(f"by 50% token generation (AUROC 0.75) but not yet")
print(f"committed at 10% (AUROC 0.51)")
print("="*55)